[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C04/E04-1_Reduction_Rebuts.ipynb)

# Réduction des rebuts d'une ligne : mettre les variables à l'échelle dans un pipeline

## Le contexte

Une ligne d'usinage produit des lots de pièces. Pour chaque lot, on relève la machine, le régime de production, l'équipe, la présence de fumée d'huile, le temps de cycle, la taille du lot, l'énergie consommée par kilogramme usiné et la température moyenne. À la fin du lot, une partie des pièces est rebutée. Le responsable de production voudrait prévoir le taux de rebuts d'un lot d'après ses conditions de production, pour agir avant qu'il ne grimpe.

Ces grandeurs n'ont ni la même unité ni le même ordre de grandeur : une taille de lot se compte en dizaines, une énergie en unités, une fumée est un oui ou un non. Les modèles qui comparent des lots par une distance voient surtout la grandeur aux plus grands nombres. La mise à l'échelle règle ce biais, à condition de se faire dans un pipeline, sur l'apprentissage seul.

C'est un problème de régression : la grandeur à prédire est un nombre, un pourcentage.

Défi : prédire `taux_rebuts_pct` et obtenir un R² supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ est celui des plus proches voisins, qui compare les lots par une distance, et ses mesures gardent leurs unités brutes ; il atteint un R² de 0,427 : à vous de faire mieux, en mettant les variables à l'échelle dans un pipeline. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Reduction_Rebuts.xlsx`, dans le dossier `donnees` du dépôt : 244 lots, 14 colonnes. Une ligne est un lot.

| Colonne | Signification | Unité |
|---|---|---|
| `type_machine` | type de machine de la ligne | catégorie |
| `regime_production` | régime : `semaine` ou `weekend` | catégorie |
| `periode_equipe` | période de l'équipe | catégorie |
| `operateur_genre` | genre de l'opérateur, écarté à l'étape 3 | catégorie |
| `fumee_huile`, `fumee_active` | fumée d'huile présente (`Yes` ou `No`), et la même information codée `1` ou `0` | catégorie, 0 ou 1 |
| `temps_cycle_min` | temps de cycle moyen | min |
| `taille_lot` | nombre de pièces du lot | pièces |
| `pieces_rebutees` | nombre de pièces rebutées | pièces |
| `taux_rebuts_pct` | cible : taux de rebuts | % |
| `energie_specifique_kwh_kg` | énergie par kilogramme usiné | kWh/kg |
| `temperature_moy_°C` | température moyenne | °C |
| `indice_performance_qualite` | indice de performance, construit | sans unité |
| `classe_qualite` | classe de qualité du lot, de `A` à `D` | catégorie |

Origine : jeu public transformé. Ce support pédagogique réinterprète les colonnes d'origine en grandeurs de production ; les rebuts, l'indice et la classe ont été construits. Il enseigne une méthode, et aucune conclusion de production n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides et ses colonnes de catégories avant toute modélisation : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Reduction_Rebuts.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Reduction_Rebuts.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "taux_rebuts_pct"
print("dimensions :", df.shape)
print("cases vides au total :", int(df.isna().sum().sum()))
for colonne in ["type_machine", "regime_production", "periode_equipe", "fumee_huile"]:
    print(colonne, df[colonne].value_counts().to_dict())
df.head(3)

Lecture du résultat : 244 lots, 14 colonnes, aucune case vide. Cinq colonnes sont des catégories, que les modèles ne lisent pas en l'état. Les lots de semaine sont minoritaires par rapport aux lots de week-end, et la fumée d'huile est présente dans une minorité de lots.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Reduction_Rebuts.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : mettre le test de côté

Un quart des lots est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec la cible. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent. La mise à l'échelle ne sera ajustée qu'après ce découpage, sur l'apprentissage seul.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=cible)
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=GRAINE)
print("apprentissage :", len(X_train), "lots")
print("test :", len(X_test), "lots")

### Étape 3 : écarter ce qui se déduit de la cible, et ce qui décrit une personne

Trois colonnes semblent proches de la cible : le nombre de pièces rebutées, l'indice de performance et la classe de qualité. On vérifie, sur l'apprentissage seul, si elles s'en déduisent, puis on compare les deux colonnes de fumée.

In [ ]:
ecart_taux = (X_train["pieces_rebutees"] / X_train["taille_lot"] * 100 - y_train).abs().max()
ecart_indice = (X_train["indice_performance_qualite"] - (1 - y_train / 10)).abs().max()
print(f"écart maximal entre le taux et pièces rebutées / taille du lot : {ecart_taux:.3f} point")
print(f"écart maximal entre l'indice et 1 - taux / 10 : {ecart_indice:.3f}")
print(y_train.groupby(X_train["classe_qualite"]).agg(["min", "max", "count"]).to_string())
print(pd.crosstab(X_train["fumee_huile"], X_train["fumee_active"]))

ecartees = ["pieces_rebutees", "indice_performance_qualite", "classe_qualite", "fumee_active", "operateur_genre"]
X_train = X_train.drop(columns=ecartees)
X_test = X_test.drop(columns=ecartees)
categories = ["type_machine", "regime_production", "periode_equipe", "fumee_huile"]
mesures = ["temps_cycle_min", "taille_lot", "energie_specifique_kwh_kg", "temperature_moy_°C"]
print("entrées gardées :", list(X_train.columns))

Lecture du résultat : le taux est le nombre de pièces rebutées divisé par la taille du lot, à l'arrondi près ; l'indice est une fonction affine du taux ; chaque classe est une tranche de taux. Ces trois colonnes sont la cible sous une autre forme, connues seulement une fois le lot terminé : elles sont écartées. Les deux colonnes de fumée disent la même chose, l'une en texte, l'autre en 0 et 1 : on garde la première.

La colonne `operateur_genre` décrit une personne, et non le procédé : elle est écartée elle aussi. Un modèle de production ne s'appuie jamais sur le genre, l'âge ou l'origine d'une personne : il jugerait les personnes au lieu des lots, et reproduirait les biais éventuels de l'historique. Le taux de rebuts se cherche dans la machine, le régime et les réglages.

### Étape 4 : regarder les échelles

Le tableau compare, pour les quatre mesures numériques de l'apprentissage, la plage de valeurs et l'écart-type.

In [ ]:
echelles = X_train[mesures].agg(["min", "max", "std"]).T
print(echelles.round(2).to_string())

Lecture du résultat : l'écart-type va de 0,6 pour l'énergie spécifique à 45 pour la taille du lot, soit un rapport de 75 à 1. Les colonnes de catégories, une fois converties en 0 et 1, auront un écart-type de l'ordre de 0,5. Un modèle qui calcule des distances verra surtout la taille du lot.

### Étape 5 : construire le modèle de départ

Le modèle de départ est celui de la section du livre, les plus proches voisins, ici en régression : il prédit le taux d'un lot par la moyenne des taux de ses cinq voisins les plus proches. Les catégories sont converties en colonnes 0 ou 1 ; les quatre mesures numériques gardent leurs unités brutes. Il apprend sur `X_train` seul.

In [ ]:
# Votre code :

### Étape 6 : mesurer sur la base de test

Le modèle prédit les lots qu'il n'a jamais vus. Trois indicateurs sont lus : le coefficient de détermination R², et deux erreurs en points de pourcentage, la racine de l'erreur quadratique moyenne (RMSE) et l'erreur absolue moyenne (MAE).

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

y_pred = modele.predict(X_test)
score_depart = r2_score(y_test, y_pred)
print(f"R² : {score_depart:.3f}")
print(f"RMSE : {root_mean_squared_error(y_test, y_pred):.2f} point")
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f} point")
print(f"taux de rebuts moyen de la base de test : {y_test.mean():.2f} %")

Lecture du résultat : le R² est d'environ 0,43, et l'erreur absolue moyenne de 1,02 point pour un taux moyen de 3,90 %. Le modèle est faible, ce que l'étape 4 laissait prévoir : ses distances sont dominées par la taille du lot. C'est ce chiffre, mesuré sur des lots que le modèle n'a jamais vus, que l'on a le droit d'annoncer.

## Pour aller plus loin : mettre à l'échelle sans fuite, puis régler le modèle

Le modèle de départ de l'étape 5 reste le score de référence du défi. Les quatre cellules suivantes montrent la démarche complète. La première chiffre le poids de chaque colonne dans les distances, avec et sans mise à l'échelle. La deuxième compare par validation croisée, sur l'apprentissage seul, le modèle brut, les voisins mis à l'échelle et réglés, et d'autres familles. La troisième montre l'erreur à ne pas commettre : ajuster l'échelle avant de découper les plis. La quatrième mesure le modèle retenu, une seule fois, sur le test. Les valeurs sont lues dans les sorties imprimées.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

In [ ]:
# Votre code :

In [ ]:
# Votre code :

## Le score de référence

Le score publié est le R² du modèle de départ de l'étape 5 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = r2_score(y_test, modele.predict(X_test))
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE r2 = %.3f" % score)

## Synthèse

1. Une colonne qui se déduit de la cible, comme le nombre de pièces rebutées, l'indice ou la classe, est écartée des entrées ; on le vérifie sur l'apprentissage seul.
2. Une colonne qui décrit une personne, son genre, son âge ou son origine, n'entre jamais dans un modèle de production.
3. Les modèles fondés sur des distances voient surtout les grandeurs aux plus grands nombres : on les met à l'échelle. Les arbres et les forêts n'en ont pas besoin.
4. La mise à l'échelle apprend des paramètres par colonne : elle s'ajuste sur l'apprentissage seul, dans un pipeline, qui la refait dans chaque pli de la validation croisée.
5. Les réglages d'un modèle se choisissent sur l'apprentissage ; le test ne sert qu'une fois, au modèle retenu.